# 03 · Analysis

**Question:** how do the social and economic conditions of a country relate to the performance of the companies that operate there?

This notebook runs the analysis plan on the database. Notebook 02 explored the data; here we test the relationships properly, with controls and honest uncertainty.

| Section | Question | Method |
|---|---|---|
| 1 | How do the 34 countries compare? | Standardised dimension scores, rankings, radar profiles |
| 2 | Which country factors go with business dynamism and market depth? | Spearman correlation with bootstrap intervals, partial correlation |
| 3 | Which country factors go with **firm performance**? | Regression of ROA on country features + sector + size + year, errors clustered by country |
| 4 | EU vs non-EU | Same regression with an EU indicator |
| 5 | Are there country "types"? | PCA + k-means, clusters mapped and compared |
| 6 | Inside Europe: Nordic, Western, Southern, Eastern | Eurostat features by group |
| 7 | Answers and limits | |

**Reading the regressions.** The outcome is ROA in **percentage points**. Country features are standardised across the 34 countries, so a coefficient is *the change in ROA, in percentage points, for a country that is 1 standard deviation higher on that feature*. Every firm is compared with firms of its own sector, relative size and fiscal year. Standard errors are **clustered by country**, because all firms in a country share the same feature values.

> **Association, not causation.** With 34 countries, most results are leads for discussion, not proof. Where we test many features at once, we say how many survive a multiple-testing correction.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, panel, profile, profile_all = country["dim"], country["panel"], country["profile"], country["profile_all"]
feat_cols, group = country["feat_cols"], country["group"]
fy = prep.load_firms(con, country)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS

cn = (fy.groupby("iso3").agg(median_roa_pp=("roa_pp", "median"), roa_adj_pp=("roa_sector_adj_pp", "median"),
                             median_margin=("operating_margin_w", "median"), firms=("company_id", "nunique"))
      .query("firms >= @MIN_FIRMS"))
cn = cn.join(dim[["name", "region", "is_eu", "eu_group", "income_group", "small_firm_sample"]])
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} company-years of {fy.company_id.nunique():,} companies | {len(cn)} countries with a country-level outcome")

34 countries | 35 country features | 10,220 company-years of 2,590 companies | 34 countries with a country-level outcome


## 1 · How do the 34 countries compare?
Seven **dimension scores** summarise the country features. Each is the average of its standardised indicators (flipped where lower is better), then converted to a **percentile (0–100) among the 34 countries**. The weights are equal and the groupings are a judgement call: this is a descriptive profile, not a measured "quality" of a country.

| Dimension | Built from |
|---|---|
| Institutions | the six governance indicators (rule of law, corruption control, government effectiveness, regulatory quality, voice, political stability) |
| Prosperity | GDP per person (PPP), life expectancy |
| Innovation | R&D spend, high-tech exports, resident patents, tertiary enrolment |
| Digital | internet users, mobile subscriptions |
| Labour market | unemployment and youth unemployment (lower = better), labour participation |
| Openness & finance | trade, foreign direct investment, private credit (all % of GDP) |
| Stability | inflation and government debt (lower = better), GDP growth |

In [2]:
DIMENSIONS = {
 "Institutions": [("gov_effectiveness", 1), ("regulatory_quality", 1), ("rule_of_law", 1), ("corruption_control", 1), ("voice_accountability", 1), ("political_stability", 1)],
 "Prosperity": [("gdp_per_capita_ppp", 1), ("life_expectancy", 1)],
 "Innovation": [("rd_gdp", 1), ("hightech_exports", 1), ("patents_residents", 1), ("tertiary_enrolment", 1)],
 "Digital": [("internet_users", 1), ("mobile_subscriptions", 1)],
 "Labour market": [("unemployment", -1), ("youth_unemployment", -1), ("labour_participation", 1)],
 "Openness & finance": [("trade_gdp", 1), ("fdi_gdp", 1), ("private_credit_gdp", 1)],
 "Stability": [("inflation", -1), ("gov_debt_gdp", -1), ("gdp_growth", 1)],
}
z = (profile - profile.mean()) / profile.std()
dim_z = pd.DataFrame({d: pd.concat([z[f] * s for f, s in items], axis=1).mean(axis=1) for d, items in DIMENSIONS.items()})
dim_pct = dim_z.rank(pct=True) * 100
overall = dim_z.mean(axis=1)
rank = pd.DataFrame({"country": names, "overall rank": overall.rank(ascending=False).astype(int)}).join(dim_pct.round(0)).join(cn[["firms", "median_roa_pp"]])
rank = rank.sort_values("overall rank")
rank

,country,overall rank,Institutions,Prosperity,Innovation,Digital,Labour market,Openness & finance,Stability,firms,median_roa_pp
iso3,,,,,,,,,,,
SGP,Singapore,1,82.000,100.000,97.000,100.000,100.000,100.000,59.000,100,2.948
KOR,South Korea,2,59.000,76.000,100.000,94.000,85.000,79.000,91.000,99,2.561
DNK,Denmark,3,97.000,82.000,59.000,85.000,71.000,94.000,85.000,100,2.945
IRL,Ireland,4,71.000,94.000,53.000,44.000,68.000,68.000,100.000,43,4.535
AUS,Australia,5,85.000,88.000,85.000,53.000,88.000,62.000,82.000,100,4.277
SWE,Sweden,6,91.000,91.000,76.000,82.000,38.000,88.000,74.000,100,3.874
JPN,Japan,7,74.000,79.000,88.000,97.000,94.000,82.000,3.000,100,4.366
NLD,Netherlands,8,88.000,85.000,68.000,71.000,97.000,18.000,76.000,80,2.455
LUX,Luxembourg,9,94.000,97.000,3.000,91.000,41.000,74.000,88.000,10,3.500


In [3]:
order = rank.index.tolist()
fig = px.imshow(dim_pct.loc[order].rename(index=names), color_continuous_scale="RdYlBu", zmin=0, zmax=100, aspect="auto", text_auto=".0f",
                labels=dict(color="percentile"), title="Dimension scores (percentile among the 34 countries), ordered by overall score")
fig.update_layout(height=900)
fig.show()

**Radar profile.** Pick a country in the dropdown; it is drawn against the median of the EU and of the non-EU countries.

In [4]:
cats = list(DIMENSIONS)
eu_med = dim_pct[dim.loc[dim_pct.index, "is_eu"]].median()
non_med = dim_pct[~dim.loc[dim_pct.index, "is_eu"]].median()
fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=eu_med[cats].tolist() + [eu_med[cats[0]]], theta=cats + [cats[0]], name="EU median", line=dict(color="#4C78A8", dash="dot")))
fig.add_trace(go.Scatterpolar(r=non_med[cats].tolist() + [non_med[cats[0]]], theta=cats + [cats[0]], name="non-EU median", line=dict(color="#F58518", dash="dot")))
first = rank.index[0]
for iso in rank.index:
    fig.add_trace(go.Scatterpolar(r=dim_pct.loc[iso, cats].tolist() + [dim_pct.loc[iso, cats[0]]], theta=cats + [cats[0]], name=names[iso],
                                  fill="toself", opacity=.5, line=dict(color="#54A24B"), visible=(iso == first)))
buttons = [dict(label=names[iso], method="update", args=[{"visible": [True, True] + [i == k for i in range(len(rank))]}, {"title": f"{names[iso]} vs EU / non-EU median"}])
           for k, iso in enumerate(rank.index)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=1.15, xanchor="left")], polar=dict(radialaxis=dict(range=[0, 100])), height=520,
                  title=f"{names[first]} vs EU / non-EU median")
fig.show()

## 2 · Which country factors go with business dynamism and market depth?
Two outcomes that don't need firm data: **new business density** (new companies per 1,000 people) and **stock market value as % of GDP** (22 countries only).
For each, the strongest country features by Spearman ρ with a **bootstrap 95% interval** (resampling countries 2,000 times) and the **partial ρ controlling for GDP per person**. The partial correlation asks whether a feature matters beyond simply "richer countries do everything better".

In [5]:
rng = np.random.default_rng(0)
def rho(x, y):
    return stats.spearmanr(x, y)[0]
def boot_ci(x, y, n=2000):
    idx = rng.integers(0, len(x), (n, len(x)))
    r = [rho(x[i], y[i]) for i in idx]
    return np.nanpercentile(r, [2.5, 97.5])
def partial_rho(x, y, c):
    rx, ry, rc = (stats.rankdata(v) for v in (x, y, c))
    res = lambda a: a - np.polyval(np.polyfit(rc, a, 1), rc)
    return stats.pearsonr(res(rx), res(ry))[0]

rows = []
for out in ["new_business_density", "market_cap_gdp"]:
    d = profile_all[feat_cols + [out]].dropna(subset=[out])
    ok = {f: d[f].notna() & d[out].notna() for f in feat_cols}          # use the countries that have both values
    cand = {f: rho(d[f][ok[f]], d[out][ok[f]]) for f in feat_cols if ok[f].sum() >= 12}
    for f in sorted(cand, key=lambda k: -abs(cand[k]))[:8]:
        m = ok[f]
        x, y = d[f][m].values, d[out][m].values
        lo, hi = boot_ci(x, y)
        mc = m & d.gdp_per_capita_ppp.notna()
        rows.append({"outcome": out, "feature": f, "n": int(m.sum()), "ρ": cand[f], "95% CI low": lo, "95% CI high": hi,
                     "partial ρ | GDP per person": partial_rho(d[f][mc].values, d[out][mc].values, d.gdp_per_capita_ppp[mc].values) if f != "gdp_per_capita_ppp" else np.nan})
levelA = pd.DataFrame(rows)
levelA["CI excludes 0"] = (levelA["95% CI low"] > 0) | (levelA["95% CI high"] < 0)
levelA.set_index(["outcome", "feature"])

n      ρ  95% CI low  95% CI high  partial ρ | GDP per person  CI excludes 0
outcome              feature                                                                                            
new_business_density population            33 -0.436      -0.743       -0.057                      -0.398           True
                     gdp_usd               33 -0.378      -0.693       -0.002                      -0.381           True
                     corruption_control    33  0.368       0.019        0.661                       0.411           True
                     internet_users        33  0.366      -0.023        0.679                       0.365          False
                     patents_residents     33 -0.354      -0.658        0.025                      -0.367          False
                     gov_debt_gdp          33 -0.351      -0.659        0.024                      -0.348          False
                     regulatory_quality    33  0.329      -0.042        0.636                       0.350          False
                     manufacturing_share   33 -0.325      -0.638        0.048                      -0.323          False
market_cap_gdp       private_credit_gdp    20  0.768       0.467        0.934                       0.845           True
                     trade_gdp             22 -0.504      -0.828       -0.053                      -0.655           True
                     gdp_usd               22  0.501       0.088        0.797                       0.510           True
                     gini                  14  0.486      -0.128        0.877                       0.662          False
                     inflation             22 -0.459      -0.736       -0.019                      -0.438           True
                     urban_share           22  0.459       0.002        0.789                       0.451           True
                     labour_participation  22  0.444       0.018        0.793                       0.418           True
                     patents_residents     22  0.431       0.005        0.740                       0.425           True

In [6]:
d = profile_all.join(dim[["name", "region"]])
fig = make_subplots(rows=1, cols=2, subplot_titles=("New business density vs corruption control", "Market value (% GDP) vs private credit (% GDP)"))
for k, (x, y) in enumerate([("corruption_control", "new_business_density"), ("private_credit_gdp", "market_cap_gdp")], 1):
    dd = d.dropna(subset=[x, y])
    for reg, g in dd.groupby("region"):
        fig.add_trace(go.Scatter(x=g[x], y=g[y], mode="markers+text", text=g.index, textposition="top center", textfont_size=8, name=reg,
                                 marker=dict(size=8), showlegend=(k == 1), legendgroup=reg), row=1, col=k)
    fig.update_xaxes(title_text=x, row=1, col=k); fig.update_yaxes(title_text=y, row=1, col=k)
fig.update_layout(height=480)
fig.show()

## 3 · Which country factors go with firm performance?
### 3.1 The model
For each company-year:

$$\text{ROA}_{i,t} = \alpha + \beta\cdot \text{feature}_{c(i),\,t-1} + \text{sector}_i + \text{size rank}_{i,t} + \text{fiscal year}_t + \varepsilon_{i,t}$$

- **ROA** is net income / total assets in percentage points, winsorised at the 1st and 99th percentile.
- **feature** is the country feature of the *previous* year, standardised across countries.
- **Controls:** sector (12 groups), **size rank** (the company's percentile of total assets within its country and year; this avoids exchange rates), and fiscal-year dummies that absorb the common macro cycle.
- **Errors clustered by country** (34 clusters).

First, how much do the controls alone explain?

In [7]:
CONTROLS = "C(sector) + size_rank + C(fiscal_year)"
def fit(formula, data, cluster="iso3"):
    return smf.ols(formula, data).fit(cov_type="cluster", cov_kwds={"groups": data[cluster].values})

d0 = fy.dropna(subset=["roa_pp", "size_rank"])
base = smf.ols(f"roa_pp ~ {CONTROLS}", d0).fit()
print(f"Controls only (sector, size rank, fiscal year): n = {int(base.nobs):,}, R² = {base.rsquared:.3f}, adjusted R² = {base.rsquared_adj:.3f}")
sz = fit(f"roa_pp ~ {CONTROLS}", d0).summary2().tables[1].loc["size_rank"]
print(f"Size rank coefficient: {sz['Coef.']:+.2f} pp ROA from the smallest to the largest company in its country (p = {sz['P>|z|']:.3f})")

Controls only (sector, size rank, fiscal year): n = 10,220, R² = 0.029, adjusted R² = 0.028
Size rank coefficient: +1.89 pp ROA from the smallest to the largest company in its country (p = 0.130)


### 3.2 One feature at a time
Many country features are highly correlated, so each is tested **alone** (35 regressions), always with the same controls. The table shows the coefficient (pp of ROA per +1 SD), its 95% interval, the raw p-value, the **Benjamini–Hochberg q-value** (corrects for testing 35 features: the share of "discoveries" expected to be false at that threshold) and how much adjusted R² improves over the controls-only model.

In [8]:
rows = []
for f in feat_cols:
    d = fy.dropna(subset=["roa_pp", "size_rank", f + "_z"])
    m = fit(f"roa_pp ~ {f}_z + {CONTROLS}", d)
    ci = m.conf_int().loc[f + "_z"]
    rows.append({"feature": f, "group": group[f], "coef (pp ROA per +1 SD)": m.params[f + "_z"], "ci_low": ci[0], "ci_high": ci[1],
                 "p": m.pvalues[f + "_z"], "n firm-years": int(m.nobs), "Δ adj. R²": m.rsquared_adj - smf.ols(f"roa_pp ~ {CONTROLS}", d).fit().rsquared_adj})
one = pd.DataFrame(rows).set_index("feature")
one["q (BH)"] = multipletests(one.p, method="fdr_bh")[1]
one = one.sort_values("p")
print(f"{(one.p < 0.05).sum()} of {len(one)} features have p < 0.05 (about {0.05 * len(one):.1f} expected by chance); {(one['q (BH)'] < 0.05).sum()} survive the multiple-testing correction (q < 0.05), {(one['q (BH)'] < 0.10).sum()} at q < 0.10.")
one.head(15)

21 of 35 features have p < 0.05 (about 1.8 expected by chance); 20 survive the multiple-testing correction (q < 0.05), 21 at q < 0.10.


,group,coef (pp ROA per +1 SD),ci_low,ci_high,p,n firm-years,Δ adj. R²,q (BH)
feature,,,,,,,,
internet_users,infrastructure,-1.248,-1.691,-0.805,0.000,10220,0.016,0.000
population,social,1.482,0.869,2.094,0.000,10220,0.022,0.000
corruption_control,governance,-1.053,-1.540,-0.566,0.000,10220,0.015,0.000
gdp_per_capita_ppp,economy,-1.115,-1.648,-0.581,0.000,10220,0.016,0.000
self_employed,labour,0.950,0.469,1.431,0.000,10220,0.013,0.001
rule_of_law,governance,-0.957,-1.458,-0.456,0.000,10220,0.013,0.001
regulatory_quality,governance,-0.942,-1.455,-0.429,0.000,10220,0.012,0.002
gov_effectiveness,governance,-0.853,-1.322,-0.385,0.000,10220,0.010,0.002
life_expectancy,social,-0.989,-1.556,-0.422,0.001,10220,0.014,0.002


In [9]:
t = one.sort_values("coef (pp ROA per +1 SD)")
fig = go.Figure(go.Scatter(x=t["coef (pp ROA per +1 SD)"], y=t.index, mode="markers", marker=dict(size=7, color=np.where(t.p < 0.05, "#E45756", "#9aa0a6")),
                           error_x=dict(type="data", symmetric=False, array=t.ci_high - t["coef (pp ROA per +1 SD)"], arrayminus=t["coef (pp ROA per +1 SD)"] - t.ci_low, thickness=1)))
fig.add_vline(x=0, line_color="grey")
fig.update_layout(height=880, xaxis_title="change in ROA (percentage points) per +1 SD of the country feature, 95% CI (red = p < 0.05)",
                  title="Country features and firm ROA: one feature at a time, with controls, clustered by country")
fig.show()

### 3.3 A second opinion at country level
The firm-level model has 10,000 rows but only 34 independent countries. As a check, regress each **country's sector-adjusted median ROA** on the feature (34 observations, robust errors). It answers a cleaner question with fewer observations. Agreement in sign between the two is a sanity check, not a second proof (same data).

In [10]:
rows = []
for f in feat_cols:
    d = cn.join(profile[[f]]).dropna(subset=[f])
    d = d.assign(x=(d[f] - profile[f].mean()) / profile[f].std())
    m = smf.ols("roa_adj_pp ~ x", d).fit(cov_type="HC3")
    rows.append({"feature": f, "country-level coef (pp)": m.params["x"], "p (country level)": m.pvalues["x"], "n countries": int(m.nobs),
                 "R² (country level)": m.rsquared})
ctry = pd.DataFrame(rows).set_index("feature")
cmp_ = one.join(ctry)
cmp_["same sign"] = np.sign(cmp_["coef (pp ROA per +1 SD)"]) == np.sign(cmp_["country-level coef (pp)"])
cmp_["q (country level)"] = multipletests(cmp_["p (country level)"], method="fdr_bh")[1]
print(f"Same sign in both models for {cmp_['same sign'].sum()} of {len(cmp_)} features. Correlation of the two sets of coefficients: ρ = {stats.spearmanr(cmp_['coef (pp ROA per +1 SD)'], cmp_['country-level coef (pp)'])[0]:.2f}")
cmp_[["coef (pp ROA per +1 SD)", "p", "q (BH)", "country-level coef (pp)", "p (country level)", "q (country level)", "R² (country level)", "same sign"]].sort_values("p").head(12)

Same sign in both models for 33 of 35 features. Correlation of the two sets of coefficients: ρ = 0.94


,coef (pp ROA per +1 SD),p,q (BH),country-level coef (pp),p (country level),q (country level),R² (country level),same sign
feature,,,,,,,,
internet_users,-1.248,0.000,0.000,-0.628,0.001,0.018,0.356,True
population,1.482,0.000,0.000,0.516,0.034,0.100,0.240,True
corruption_control,-1.053,0.000,0.000,-0.451,0.003,0.023,0.184,True
gdp_per_capita_ppp,-1.115,0.000,0.000,-0.508,0.019,0.062,0.233,True
self_employed,0.950,0.000,0.001,0.490,0.115,0.269,0.217,True
rule_of_law,-0.957,0.000,0.001,-0.423,0.007,0.046,0.161,True
regulatory_quality,-0.942,0.000,0.002,-0.430,0.012,0.046,0.167,True
gov_effectiveness,-0.853,0.000,0.002,-0.417,0.001,0.021,0.157,True
life_expectancy,-0.989,0.001,0.002,-0.597,0.002,0.021,0.322,True


### 3.4 Several features together
Putting related features together makes coefficients unstable, so we choose a short list of **representative** features, one per theme, and drop any with a high **variance inflation factor** (VIF, above 5 means the feature is largely explained by the others).

In [11]:
cand = ["gdp_per_capita_ppp", "rule_of_law", "trade_gdp", "rd_gdp", "unemployment", "private_credit_gdp", "inflation", "gdp_growth", "pop_65plus"]
keep = cand.copy()
while True:
    X = profile[keep].dropna()                      # countries complete on these features
    X = (X - X.mean()) / X.std(); X["const"] = 1.0
    vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(len(keep))], index=keep)
    if vif.max() <= 5:
        break
    print(f"dropping {vif.idxmax()} (VIF {vif.max():.1f})")
    keep.remove(vif.idxmax())
print("kept:", keep, f"| VIF computed on {len(X)} countries with complete data")
vif.rename("VIF").to_frame()

dropping gdp_per_capita_ppp (VIF 7.5)
kept: ['rule_of_law', 'trade_gdp', 'rd_gdp', 'unemployment', 'private_credit_gdp', 'inflation', 'gdp_growth', 'pop_65plus'] | VIF computed on 29 countries with complete data


,VIF
rule_of_law,3.787
trade_gdp,2.654
rd_gdp,3.185
unemployment,1.827
private_credit_gdp,3.063
inflation,2.163
gdp_growth,1.330
pop_65plus,2.551


In [12]:
d = fy.dropna(subset=["roa_pp", "size_rank"] + [f + "_z" for f in keep])
multi = fit("roa_pp ~ " + " + ".join(f + "_z" for f in keep) + f" + {CONTROLS}", d)
tab = pd.DataFrame({"coef": multi.params, "ci_low": multi.conf_int()[0], "ci_high": multi.conf_int()[1], "p": multi.pvalues}).loc[[f + "_z" for f in keep]]
tab.index = keep
wald = multi.wald_test(", ".join(f"{f}_z = 0" for f in keep), scalar=True)
base_d = smf.ols(f"roa_pp ~ {CONTROLS}", d).fit()
print(f"n = {int(multi.nobs):,} company-years, {d.iso3.nunique()} countries | adjusted R² {multi.rsquared_adj:.3f} vs {base_d.rsquared_adj:.3f} with controls only | joint Wald test of all country features: p = {float(wald.pvalue):.3f}")
tab

n = 8,568 company-years, 33 countries | adjusted R² 0.047 vs 0.026 with controls only | joint Wald test of all country features: p = 0.000


,coef,ci_low,ci_high,p
rule_of_law,-0.422,-1.111,0.266,0.229
trade_gdp,-1.011,-1.888,-0.134,0.024
rd_gdp,-0.227,-0.782,0.328,0.422
unemployment,-0.128,-0.466,0.210,0.457
private_credit_gdp,0.791,0.106,1.475,0.024
inflation,0.366,0.095,0.636,0.008
gdp_growth,0.093,-0.126,0.311,0.406
pop_65plus,-0.216,-0.897,0.464,0.533


### 3.5 Do the results hold up?
For the three strongest single features, re-estimate after removing parts of the sample. A real pattern should keep its sign and roughly its size.

In [13]:
top3 = one.index[:3].tolist()
scen = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
rows = []
for f in top3:
    for name, flt in scen.items():
        d = flt(fy).dropna(subset=["roa_pp", "size_rank", f + "_z"])
        try:
            m = fit(f"roa_pp ~ {f}_z + {CONTROLS}", d)
            rows.append({"feature": f, "sample": name, "coef": m.params[f + "_z"], "p": m.pvalues[f + "_z"], "countries": d.iso3.nunique(), "firm-years": len(d)})
        except Exception as e:
            rows.append({"feature": f, "sample": name, "coef": np.nan, "p": np.nan, "countries": d.iso3.nunique(), "firm-years": len(d)})
rob = pd.DataFrame(rows)
fig = px.bar(rob, x="coef", y="sample", color="feature", barmode="group", orientation="h", hover_data=["p", "countries"],
             title="Coefficient (pp ROA per +1 SD) for the three strongest features under different samples", height=520)
fig.add_vline(x=0, line_color="grey"); fig.update_layout(yaxis_autorange="reversed")
fig.show()
rob.pivot(index="sample", columns="feature", values="coef").loc[list(scen)]

feature,corruption_control,internet_users,population
sample,,,
all firms,-1.053,-1.248,1.482
excluding financials & real estate,-1.241,-1.477,1.878
excluding small-sample countries,-1.045,-1.231,1.595
"excluding USA, China, Japan",-1.115,-1.381,1.431
fiscal years 2023 and later,-1.103,-1.432,1.585
EU countries only,-0.974,-1.339,0.255
non-EU countries only,-0.618,-0.859,1.511


In [14]:
# leave-one-country-out for the strongest feature: is one country driving it?
f = top3[0]
loo = []
for iso in fy.iso3.unique():
    d = fy[fy.iso3 != iso].dropna(subset=["roa_pp", "size_rank", f + "_z"])
    loo.append({"left out": names[iso], "coef": fit(f"roa_pp ~ {f}_z + {CONTROLS}", d).params[f + "_z"]})
loo = pd.DataFrame(loo).sort_values("coef")
full_coef = one.loc[f, "coef (pp ROA per +1 SD)"]
print(f"{f}: coefficient with all countries = {full_coef:+.2f}; leaving out one country at a time it ranges from {loo.coef.min():+.2f} ({loo.iloc[0]['left out']}) to {loo.coef.max():+.2f} ({loo.iloc[-1]['left out']}).")
px.bar(loo, x="left out", y="coef", title=f"Leave-one-country-out: {f}", height=380).add_hline(y=full_coef, line_dash="dot", line_color="red")

internet_users: coefficient with all countries = -1.25; leaving out one country at a time it ranges from -1.37 (United States) to -1.06 (India).


### 3.6 Is the inference too optimistic? Wild cluster bootstrap
With only 34 clusters (countries), the usual cluster-robust standard errors are known to be **too small**, which makes p-values too optimistic. The standard remedy is the **wild cluster bootstrap**: re-draw the residuals country by country with random signs (under the assumption that the coefficient is zero) 1,999 times, and see how often a t-statistic as large as the observed one appears. The bootstrap p-value is the one to quote.

In [15]:
import patsy
def wild_cluster_p(focal, data, extra="", B=1999, seed=0):
    d = data.dropna(subset=["roa_pp", "size_rank", focal])
    formula = f"roa_pp ~ {focal}" + (f" + {extra}" if extra else "") + f" + {CONTROLS}"
    y, X = patsy.dmatrices(formula, d, return_type="dataframe")
    g = pd.factorize(d.iso3)[0]
    o = np.argsort(g, kind="stable")
    Xv, yv, gv = X.values[o], y.values[o, 0], g[o]
    k = list(X.columns).index(focal)
    XtXi = np.linalg.inv(Xv.T @ Xv)
    starts = np.r_[0, np.flatnonzero(np.diff(gv)) + 1]
    sizes_ = np.diff(np.r_[starts, len(yv)])
    G, (N, K) = len(starts), Xv.shape
    adj = G / (G - 1) * (N - 1) / (N - K)
    def tstat(yy):
        beta = XtXi @ (Xv.T @ yy)
        u = yy - Xv @ beta
        S = np.add.reduceat(Xv * u[:, None], starts, axis=0)
        V = adj * XtXi @ (S.T @ S) @ XtXi
        return beta[k], beta[k] / np.sqrt(V[k, k])
    b_obs, t_obs = tstat(yv)
    Xr = np.delete(Xv, k, axis=1)                       # restricted model: focal coefficient = 0
    fit_r = Xr @ np.linalg.lstsq(Xr, yv, rcond=None)[0]
    res_r = yv - fit_r
    rng = np.random.default_rng(seed)
    hits = sum(abs(tstat(fit_r + np.repeat(rng.choice([-1.0, 1.0], G), sizes_) * res_r)[1]) >= abs(t_obs) for _ in range(B))
    return b_obs, t_obs, (hits + 1) / (B + 1), G

rows = []
for f in one.index[:6]:
    b, t, pw, G = wild_cluster_p(f + "_z", fy)
    rows.append({"term": f, "coef (pp ROA)": b, "t (cluster-robust)": t, "p (cluster-robust)": one.loc[f, "p"], "p (wild bootstrap)": pw, "clusters": G})
fy["eu"] = fy.iso3.map(dim.is_eu).astype(int)
b, t, pw, G = wild_cluster_p("eu", fy)
rows.append({"term": "EU vs non-EU (controls only)", "coef (pp ROA)": b, "t (cluster-robust)": t, "p (cluster-robust)": np.nan, "p (wild bootstrap)": pw, "clusters": G})
wb = pd.DataFrame(rows).set_index("term")
wb

,coef (pp ROA),t (cluster-robust),p (cluster-robust),p (wild bootstrap),clusters
term,,,,,
internet_users,-1.248,-5.524,0.000,0.005,34
population,1.482,4.744,0.000,0.002,34
corruption_control,-1.053,-4.237,0.000,0.001,34
gdp_per_capita_ppp,-1.115,-4.094,0.000,0.009,34
self_employed,0.950,3.874,0.000,0.109,34
rule_of_law,-0.957,-3.742,0.000,0.001,34
EU vs non-EU (controls only),-2.284,-3.965,NaN,0.001,34


## 4 · EU vs non-EU
Do EU-based firms differ in ROA after accounting for sector, size and year? And does the answer change once we also account for how rich the country is? A raw gap can disappear if it is really an income gap.

In [16]:
fy2 = fy.assign(eu=fy.iso3.map(dim.is_eu).astype(int))
d = fy2.dropna(subset=["roa_pp", "size_rank", "gdp_per_capita_ppp_z"])
m1 = fit(f"roa_pp ~ eu + {CONTROLS}", d)
m2 = fit(f"roa_pp ~ eu + gdp_per_capita_ppp_z + {CONTROLS}", d)
res = pd.DataFrame({"EU effect (pp ROA)": [m1.params["eu"], m2.params["eu"]], "ci_low": [m1.conf_int().loc["eu", 0], m2.conf_int().loc["eu", 0]],
                    "ci_high": [m1.conf_int().loc["eu", 1], m2.conf_int().loc["eu", 1]], "p": [m1.pvalues["eu"], m2.pvalues["eu"]]},
                   index=["controls only", "+ GDP per person"])
print(f"n = {len(d):,} company-years; EU firms: {int(d.eu.sum()):,}, non-EU: {int((1 - d.eu).sum()):,}")
res

n = 10,220 company-years; EU firms: 5,121, non-EU: 5,099


,EU effect (pp ROA),ci_low,ci_high,p
controls only,-2.284,-3.412,-1.155,0.000
+ GDP per person,-1.592,-2.951,-0.234,0.022


## 5 · Are there country "types"?
Countries are clustered on their standardised features. To reduce noise from the 35 overlapping features, the clustering runs on the **principal components** that together explain about 80% of the variance. The number of clusters (3 to 5) is the one with the best **silhouette score** (how well separated the clusters are; above 0.25 means some real structure, below 0.2 is weak), **among the options where every cluster has at least 3 countries**: a one- or two-country cluster is an outlier, not a type.

In [17]:
zz = ((profile - profile.mean()) / profile.std()).fillna(0)
U, S, Vt = np.linalg.svd(zz.values, full_matrices=False)
expl = np.cumsum(S**2 / (S**2).sum())
npc = int(np.searchsorted(expl, 0.80) + 1)
pcs = U[:, :npc] * S[:npc]
sil, labs = {}, {}
for k in range(3, 6):
    km_ = KMeans(k, n_init=50, random_state=0).fit(pcs)
    sil[k], labs[k] = silhouette_score(pcs, km_.labels_), km_.labels_
valid = [k for k in sil if np.bincount(labs[k]).min() >= 3]
best_k = max(valid or list(sil), key=lambda k: sil[k])
lab = pd.Series(labs[best_k], index=profile.index)
sizes = {k: sorted(np.bincount(labs[k]).tolist(), reverse=True) for k in sil}
# number clusters from richest (1) to poorest by mean GDP per person
order = profile.gdp_per_capita_ppp.groupby(lab).mean().sort_values(ascending=False).index
lab = lab.map({c: i + 1 for i, c in enumerate(order)})
print(f"{npc} principal components used ({expl[npc-1]:.0%} of variance). Silhouette by k: " + ", ".join(f"k={k}: {v:.2f}" for k, v in sil.items()) + f" → k = {best_k}")
print("Cluster sizes by k:", sizes, "| options with every cluster ≥ 3 countries:", valid)
members = pd.DataFrame({"cluster": lab, "country": names}).sort_values(["cluster", "country"])
members.groupby("cluster").country.apply(lambda s: ", ".join(s)).to_frame("countries")

7 principal components used (82% of variance). Silhouette by k: k=3: 0.22, k=4: 0.25, k=5: 0.25 → k = 4
Cluster sizes by k: {3: [15, 13, 6], 4: [13, 13, 5, 3], 5: [14, 13, 5, 1, 1]} | options with every cluster ≥ 3 countries: [3, 4]


,countries
cluster,
1,"Ireland, Luxembourg, Singapore"
2,"Australia, Austria, Belgium, Canada, Denmark, ..."
3,"Chile, Czechia, Estonia, Greece, Hungary, Ital..."
4,"Brazil, China, India, Indonesia, Mexico"


In [18]:
mp = pd.DataFrame({"iso3": lab.index, "cluster": lab.astype(str).values, "country": names[lab.index].values})
fig = px.choropleth(mp, locations="iso3", color="cluster", hover_name="country", category_orders={"cluster": [str(i) for i in range(1, best_k + 1)]},
                    color_discrete_sequence=px.colors.qualitative.Set2, title="Country clusters (1 = highest GDP per person)")
fig.update_geos(fitbounds="locations", projection_type="natural earth"); fig.update_layout(height=500, margin=dict(l=0, r=0, t=50, b=0))
fig.show()
key = ["gdp_per_capita_ppp", "rule_of_law", "unemployment", "inflation", "trade_gdp", "rd_gdp", "internet_users", "pop_65plus", "private_credit_gdp", "gdp_growth"]
prof_orig = profile.copy()
prof_orig["gdp_per_capita_ppp"] = np.exp(prof_orig.gdp_per_capita_ppp)      # back to dollars
summary = prof_orig[key].groupby(lab).mean().round(2)
summary.insert(0, "countries", lab.value_counts().sort_index())
summary

,countries,gdp_per_capita_ppp,rule_of_law,unemployment,inflation,trade_gdp,rd_gdp,internet_users,pop_65plus,private_credit_gdp,gdp_growth
1,3,"123,348.250",1.590,4.810,3.060,322.080,1.440,94.960,13.910,66.350,3.650
2,13,"59,880.440",1.560,5.250,3.320,87.890,3.050,92.510,20.200,126.450,1.400
3,13,"38,395.630",0.750,9.100,5.680,105.030,1.210,84.570,19.120,63.220,1.710
4,5,"16,164.830",-0.430,5.830,4.240,46.790,1.280,67.530,8.810,71.360,3.130


**Does firm performance differ between the country types?** Country median ROA by cluster, and a firm-level regression with cluster dummies (cluster 1 = reference).

In [19]:
cc = cn.join(lab.rename("cluster")).dropna(subset=["cluster"])
cc["cluster"] = cc.cluster.astype(int)
kw = stats.kruskal(*[g.roa_adj_pp.values for _, g in cc.groupby("cluster")])
fig = px.box(cc, x="cluster", y="median_roa_pp", points="all", hover_name="name", title=f"Median ROA by country cluster (Kruskal–Wallis p = {kw.pvalue:.3f}, n = {len(cc)} countries)")
fig.update_layout(height=420); fig.show()
d = fy.join(lab.rename("cluster"), on="iso3").dropna(subset=["roa_pp", "size_rank", "cluster"])
d["cluster"] = d.cluster.astype(int)
mc = fit(f"roa_pp ~ C(cluster) + {CONTROLS}", d)
pd.DataFrame({"vs cluster 1 (pp ROA)": mc.params.filter(like="C(cluster)"), "ci_low": mc.conf_int().filter(like="C(cluster)", axis=0)[0],
              "ci_high": mc.conf_int().filter(like="C(cluster)", axis=0)[1], "p": mc.pvalues.filter(like="C(cluster)")})

,vs cluster 1 (pp ROA),ci_low,ci_high,p
C(cluster)[T.2],0.988,-0.229,2.205,0.112
C(cluster)[T.3],1.863,1.304,2.423,0.000
C(cluster)[T.4],3.481,2.217,4.746,0.000


## 6 · Inside Europe: Nordic, Western, Southern, Eastern
Eurostat gives EU-only measures that the 34-country sources lack. Each is averaged over the years available (2019 onward): hourly labour cost, minimum wage in purchasing power, AI adoption by firms, business birth rate, and the economy-wide return on equity and profit share of non-financial corporations. Groups are small (3 to 9 countries), so this is descriptive.

In [20]:
EU_IND = {
 "labour_cost_eur_h": "ESTAT:lc_lci_lev:%", "min_wage_pps": "ESTAT:earn_mw_cur:currency=PPS", "ai_adoption_pct": "ESTAT:isoc_eb_ai:%",
 "business_birth_rate_pct": "ESTAT:bd_size:%ENT_BRTHR_PC%", "nfc_roe_pct": "ESTAT:nasa_10_ki:%ROE_S11%", "nfc_profit_share_pct": "ESTAT:nasa_10_ki:%B2G_B3G_RAT_S11%",
}
parts = []
for name, like in EU_IND.items():
    d = q(f"SELECT iso3, AVG(value) AS v FROM fact_country_year WHERE indicator_id LIKE '{like}' AND year >= 2019 GROUP BY 1")
    parts.append(d.set_index("iso3").v.rename(name))
eu_f = pd.concat(parts, axis=1).join(dim[["name", "eu_group"]]).join(cn[["median_roa_pp"]]).dropna(subset=["eu_group"])
eu_f["rule_of_law"] = profile.rule_of_law
cols = list(EU_IND) + ["median_roa_pp"]
tbl = eu_f.groupby("eu_group")[cols].median().round(2)
tbl.insert(0, "countries", eu_f.groupby("eu_group").size())
kwp = {c: (stats.kruskal(*[g[c].dropna().values for _, g in eu_f.groupby("eu_group")]).pvalue if eu_f[c].notna().sum() > 8 else np.nan) for c in cols}
tbl.loc["Kruskal–Wallis p"] = [np.nan] + [kwp[c] for c in cols]
tbl

,countries,labour_cost_eur_h,min_wage_pps,ai_adoption_pct,business_birth_rate_pct,nfc_roe_pct,nfc_profit_share_pct,median_roa_pp
eu_group,,,,,,,,
Eastern,7.000,13.970,814.750,7.320,11.880,16.050,45.020,3.350
Nordic,3.000,40.170,NaN,23.270,8.830,8.710,41.380,3.870
Southern,4.000,20.570,998.120,8.880,8.870,10.500,39.860,2.930
Western,7.000,41.330,"1,628.160",17.460,9.130,23.310,40.930,3.170
Kruskal–Wallis p,NaN,0.001,NaN,0.002,0.091,0.072,0.163,0.524


In [21]:
fig = make_subplots(rows=2, cols=4, subplot_titles=[c.replace("_", " ") for c in cols], vertical_spacing=0.18)
for k, c in enumerate(cols):
    for g, gg in eu_f.groupby("eu_group"):
        fig.add_trace(go.Box(y=gg[c], name=g, boxpoints="all", text=gg.name, jitter=.5, marker_size=5, showlegend=False,
                             marker_color={"Nordic": "#4C78A8", "Western": "#F58518", "Southern": "#E45756", "Eastern": "#54A24B"}[g]), row=k // 4 + 1, col=k % 4 + 1)
fig.update_layout(height=640, title="EU groups on Eurostat measures")
fig.show()

## 7 · Answers and limits
The text below is computed from the results above when the notebook runs.

In [22]:
top = one.iloc[0]
sig_q = one[one["q (BH)"] < 0.10]
m_eff = npc   # number of principal components needed for 80% of the variance of the 35 features
lines = [
 f"1. Sample: {len(fy):,} company-years, {fy.company_id.nunique():,} companies, {fy.iso3.nunique()} countries. Sector, relative size and fiscal year alone explain {base.rsquared_adj:.1%} of ROA differences (adjusted R²).",
 f"2. Strongest single feature: {one.index[0]} ({top['coef (pp ROA per +1 SD)']:+.2f} pp ROA per +1 SD, 95% CI [{top.ci_low:+.2f}, {top.ci_high:+.2f}]). It improves adjusted R² by only {top['Δ adj. R²']:.4f}. Wild-bootstrap p = {wb.loc[one.index[0], 'p (wild bootstrap)']:.3f} (cluster-robust p = {top.p:.4f}).",
 f"3. Multiple testing: {(one.p < 0.05).sum()} of {len(one)} features reach p < 0.05 and {len(sig_q)} have q < 0.10. But the 35 features are only about {m_eff} independent dimensions (principal components for 80% of variance), and the survivors are mostly one block (governance, wealth, internet use, size). Read this as ONE pattern about development level, not {len(sig_q)} separate findings.",
 f"4. Small-sample inference: with {wb['clusters'].iloc[0]} countries the wild-bootstrap p-values for the top features are {', '.join(f'{v:.3f}' for v in wb['p (wild bootstrap)'].iloc[:5])} (cluster-robust: {', '.join(f'{v:.3f}' for v in wb['p (cluster-robust)'].iloc[:5])}). They are larger, as expected, and the ones above 0.05 should not be called significant.",
 f"5. Several features together (VIF-screened {len(keep)}): joint Wald p = {float(wald.pvalue):.3f} (cluster-robust, so optimistic); adjusted R² goes from {base_d.rsquared_adj:.3f} to {multi.rsquared_adj:.3f}. All of this is a few percentage points of ROA variance.",
 f"6. Direction: {int((one['coef (pp ROA per +1 SD)'] < 0).sum())} of {len(one)} single-feature coefficients are negative. After controls, firms in more developed, more connected countries have LOWER ROA: about {abs(top['coef (pp ROA per +1 SD)']):.1f} pp per standard deviation, i.e. {abs(top['coef (pp ROA per +1 SD)']) / fy.roa_pp.median():.0%} of the typical ROA of {fy.roa_pp.median():.1f}%. That is economically visible but explains little of the spread between firms (finding 2).",
 f"7. Country-level check (n≈{len(cn)}, sector-adjusted median ROA): same sign as the firm-level model for {int(cmp_['same sign'].sum())} of {len(cmp_)} features; {int((cmp_['q (country level)'] < 0.10).sum())} have q < 0.10. This is the more defensible test.",
 f"8. EU effect: {res.iloc[0, 0]:+.2f} pp ROA with controls only (wild-bootstrap p = {wb.loc['EU vs non-EU (controls only)', 'p (wild bootstrap)']:.3f}); {res.iloc[1, 0]:+.2f} pp when also controlling for GDP per person (cluster-robust p = {res.iloc[1, 3]:.3f}). A gap like this can also come from accounting standards, from which firms Yahoo covers in each region, and from the sector mix inside the broad Yahoo sectors, so it should not be read as an 'EU effect' on its own.",
 f"9. Country types: best k = {best_k} (silhouette {sil[best_k]:.2f}; sizes {sizes[best_k]}); median ROA by cluster: Kruskal–Wallis p = {kw.pvalue:.3f}.",
]
print("\n".join(lines))

1. Sample: 10,220 company-years, 2,590 companies, 34 countries. Sector, relative size and fiscal year alone explain 2.8% of ROA differences (adjusted R²).
2. Strongest single feature: internet_users (-1.25 pp ROA per +1 SD, 95% CI [-1.69, -0.81]). It improves adjusted R² by only 0.0159. Wild-bootstrap p = 0.005 (cluster-robust p = 0.0000).
3. Multiple testing: 21 of 35 features reach p < 0.05 and 21 have q < 0.10. But the 35 features are only about 7 independent dimensions (principal components for 80% of variance), and the survivors are mostly one block (governance, wealth, internet use, size). Read this as ONE pattern about development level, not 21 separate findings.
4. Small-sample inference: with 34 countries the wild-bootstrap p-values for the top features are 0.005, 0.002, 0.001, 0.009, 0.109 (cluster-robust: 0.000, 0.000, 0.000, 0.000, 0.000). They are larger, as expected, and the ones above 0.05 should not be called significant.
5. Several features together (VIF-screened 8): j

### Limits to state in the report
- **34 countries is a small sample.** All inference about countries rests on 34 points; the 10,000 firm-years do not add independent information about country differences.
- **Association only.** No design here identifies causal effects; features also overlap strongly, so one coefficient cannot be read as "the effect of X".
- **Listed large firms**, headquarters country (not where profits are made), about four fiscal years, mixed accounting standards.
- **Yahoo sample only.** ESEF (no sector field) and the overlap between sources were not used in the regressions.
- **Size** is a within-country percentile because raw assets are in different currencies.
- **Many tests and few clusters.** P-values are shown raw, with a Benjamini–Hochberg correction, and (for the strongest) with a wild cluster bootstrap; prefer the bootstrap and the country-level check.
- **Reverse and common causes** are not ruled out: for example, listing rules and market structure affect both which firms are sampled and how they perform.

### Register the primary test (for `1.0_hypotheses.ipynb`)
The original hypothesis in one pre-agreed test: **does the country's development index (average of GDP per person, life expectancy, internet use, rule of law, government effectiveness and corruption control; previous year, SDs across countries) relate to company ROA?** No direction was predicted. Same model as above (sector, relative size, fiscal year; errors clustered by country), wild-bootstrap p-value, the seven robustness samples, and the ESEF replication. The result is saved to `results/H0.json`.

In [23]:
import utils
prim = utils.wild_cluster("dev_z", fy, outcome="roa_pp", B=1999)
rows = []
for name, flt in scen.items():
    r = utils.wild_cluster("dev_z", flt(fy), outcome="roa_pp", B=499)
    rows.append({"sample": name, "coef": r["coef"], "p (wild)": r["p_wild"], "countries": r["clusters"]})
rob0 = pd.DataFrame(rows).set_index("sample")
share0 = float((np.sign(rob0.coef) == np.sign(prim["coef"])).mean())
fe0 = prep.load_firms(con, country, source="ESEF")
rep0 = utils.wild_cluster("dev_z", fe0, outcome="roa_pp", controls="size_rank + C(fiscal_year)", B=1999)
v0 = utils.verdict(prim["coef"], prim["p_wild"], 0, share0, prim["clusters"])
lines0 = [
 f"Primary: +1 SD development → {prim['coef']:+.2f} pp ROA (95% CI {prim['ci_low']:+.2f} to {prim['ci_high']:+.2f}), wild p = {prim['p_wild']:.3f}, {prim['clusters']} countries.",
 f"Robustness: same sign in {share0:.0%} of seven samples. ESEF replication: {rep0['coef']:+.2f} pp (wild p = {rep0['p_wild']:.3f}).",
 f"VERDICT H0: {v0.upper()} (association present, direction negative; effect small relative to firm and sector differences)",
]
print("\n".join(lines0))
utils.save_result("H0", id="H0", title="Country conditions and firm performance", statement="Company ROA is related to the social and economic conditions of the country.",
                  expected_sign=0, effect=prim["coef"], ci_low=prim["ci_low"], ci_high=prim["ci_high"], effect_unit="pp ROA per +1 SD development index",
                  p_primary=prim["p_wild"], p_method="wild cluster bootstrap", n=prim["n"], clusters=prim["clusters"], robust_share=share0,
                  replication_coef=rep0["coef"], replication_p=rep0["p_wild"], replication_same_sign=bool(np.sign(rep0["coef"]) == np.sign(prim["coef"])),
                  verdict=v0, notebook="03_analysis.ipynb", notes=lines0)

Primary: +1 SD development → -1.12 pp ROA (95% CI -1.60 to -0.65), wild p = 0.001, 34 countries.
Robustness: same sign in 100% of seven samples. ESEF replication: -1.29 pp (wild p = 0.023).
VERDICT H0: SUPPORTED (association present, direction negative; effect small relative to firm and sector differences)


In [24]:
con.close()